In [30]:
import numpy as np
from Bio import Phylo
import sys
sys.path.append('../../pysimARG')
from clonal_genealogy import ClonalTree
from ClonalOrigin_ARG import ARG
from add_mutation import add_mutation
from homoplasy_index import homoplasy_index
from newick_to_tree import newick_to_tree
from G4_test import G4_test
from LD_r import LD_r
from ClonalOrigin_seq_sim import ClonalOrigin_seq_sim

In [31]:
np.random.seed(100)
clonal_tree = ClonalTree(n=10)

# Load phylo tree and convert to ClonalTree format
phylo_tree = Phylo.read("../../data/SimBac/clonal_frame.nwk", "newick")
Phylo.draw_ascii(phylo_tree)

edge, node_height = newick_to_tree(phylo_tree)
clonal_tree.edge = edge
clonal_tree.node_height = node_height
clonal_tree.height = np.max(node_height)
clonal_tree.length = np.sum(edge[:, 2])

rho_site = 0.02
theta_site = 0.05
L = 2000
delta = 300

                                                                   _______ 2
                                        __________________________|
                            ___________|                          |_______ 8
                           |           |
                           |           |___________________________________ 1
                           |
                        ___|                         _____________________ 6
                       |   |                       ,|
                       |   |                       ||  ___________________ 3
                       |   |                       ||_|
                       |   |_______________________|  |___________________ 9
  _____________________|                           |
 |                     |                           |         _____________ 5
 |                     |                           |________|
_|                     |                                    |_____________ 7
 |                  

## ClonalOrigin seq

In [32]:
ARG_sim = ARG(clonal_tree, rho_site, L, delta, L, "seq")

In [33]:
ARG_sim.node_mat

array([[1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       ...,
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.]], shape=(298, 2000))

In [34]:
node_site = add_mutation(ARG_sim, theta_site)
node_site

array([[False,  True,  True, ...,  True, False, False],
       [False,  True,  True, ...,  True, False, False],
       [False,  True,  True, ...,  True, False, False],
       ...,
       [False,  True,  True, ...,  True, False, False],
       [False,  True,  True, ...,  True, False, False],
       [False,  True,  True, ..., False, False, False]], shape=(298, 2000))

In [35]:
mat = node_site[:10, :]
mat

array([[False,  True,  True, ...,  True, False, False],
       [False,  True,  True, ...,  True, False, False],
       [False,  True,  True, ...,  True, False, False],
       ...,
       [False,  True,  True, ...,  True, False, False],
       [False,  True,  True, ...,  True, False, False],
       [False,  True,  True, ...,  True, False, False]], shape=(10, 2000))

In [36]:
s_vec = np.full(19, np.nan)
s_vec

array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan])

In [37]:
has_true = mat.any(axis=0)
has_false = ~mat.all(axis=0)
idx_seg = np.where(has_true & has_false)[0]

In [38]:
seg_near, seg_far = 0, 0
seg_20_50, seg_50_80 = 0, 0

In [39]:
ld_near, ld_far, g4_near, g4_far = 0, 0, 0, 0
ld_20_50, ld_50_80, g4_20_50, g4_50_80 = 0, 0, 0, 0
if idx_seg.size >= 2:
    for i in range(idx_seg.size - 1):
        for j in range(i + 1, idx_seg.size):
            dist_ij = idx_seg[j] - idx_seg[i]
            idx_pair = [idx_seg[i], idx_seg[j]]
            if dist_ij < L/2:
                ld_near += LD_r(mat[:, idx_pair])
                g4_near += G4_test(mat[:, idx_pair])
                seg_near += 1
            else:
                ld_far += LD_r(mat[:, idx_pair])
                g4_far += G4_test(mat[:, idx_pair])
                seg_far += 1
            if 20 <= dist_ij < 50:
                ld_20_50 += LD_r(mat[:, idx_pair])
                g4_20_50 += G4_test(mat[:, idx_pair])
                seg_20_50 += 1
            if 50 <= dist_ij <= 80:
                ld_50_80 += LD_r(mat[:, idx_pair])
                g4_50_80 += G4_test(mat[:, idx_pair])
                seg_50_80 += 1
    
    s_vec[0] = ld_near
    s_vec[1] = ld_far
    s_vec[2] = g4_near
    s_vec[3] = g4_far

    s_vec[4] = ld_near / seg_near if seg_near > 0 else 0
    s_vec[5] = ld_far /seg_far if seg_far > 0 else 0
    s_vec[6] = g4_near / seg_near if seg_near > 0 else 0
    s_vec[7] = g4_far / seg_far if seg_far > 0 else 0

    s_vec[8] = ld_20_50
    s_vec[9] = ld_50_80
    s_vec[10] = g4_20_50
    s_vec[11] = g4_50_80

    s_vec[12] = ld_20_50 / seg_20_50 if seg_20_50 > 0 else 0
    s_vec[13] = ld_50_80 / seg_50_80 if seg_50_80 > 0 else 0
    s_vec[14] = g4_20_50 / seg_20_50 if seg_20_50 > 0 else 0
    s_vec[15] = g4_50_80 / seg_50_80 if seg_50_80 > 0 else 0
else:
    s_vec[:16] = 0

In [40]:
s_vec[:8]

array([6.93209557e+03, 1.33587812e+03, 3.88900000e+03, 8.58000000e+02,
       1.49947990e-01, 9.67957478e-02, 8.41228639e-02, 6.21694080e-02])

In [41]:
s_vec[8:16]

array([4.05169143e+02, 3.65505479e+02, 5.90000000e+01, 9.60000000e+01,
       2.16899969e-01, 1.95144409e-01, 3.15845824e-02, 5.12546716e-02])

In [42]:
seg_near, seg_far, seg_20_50, seg_50_80

(46230, 13801, 1868, 1873)

In [43]:
print(seg_near + seg_far)
print(idx_seg.size * (idx_seg.size - 1) / 2)

60031
60031.0


In [44]:
s_vec[16] = homoplasy_index(clonal_tree, node_site)

count_S = idx_seg.size
s_vec[17] = count_S / L

s_vec[18] = L

s_vec

array([6.93209557e+03, 1.33587812e+03, 3.88900000e+03, 8.58000000e+02,
       1.49947990e-01, 9.67957478e-02, 8.41228639e-02, 6.21694080e-02,
       4.05169143e+02, 3.65505479e+02, 5.90000000e+01, 9.60000000e+01,
       2.16899969e-01, 1.95144409e-01, 3.15845824e-02, 5.12546716e-02,
       3.72513562e-01, 1.73500000e-01, 2.00000000e+03])

## seq sim try

In [45]:
clonalorigin_stats = np.full((200, 19), np.nan)
clonalorigin_stats

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], shape=(200, 19))

In [46]:
for idx in range(200):
    np.random.seed(100 + idx)
    s_vec = ClonalOrigin_seq_sim(clonal_tree, rho_site, theta_site, L, delta)
    clonalorigin_stats[idx, :] = s_vec

    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} segments...")

Processed 10 segments...
Processed 20 segments...
Processed 30 segments...
Processed 40 segments...
Processed 50 segments...
Processed 60 segments...
Processed 70 segments...
Processed 80 segments...
Processed 90 segments...
Processed 100 segments...
Processed 110 segments...
Processed 120 segments...
Processed 130 segments...
Processed 140 segments...
Processed 150 segments...
Processed 160 segments...
Processed 170 segments...
Processed 180 segments...
Processed 190 segments...
Processed 200 segments...
